In [8]:
import numpy as np

# Data

In [9]:
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

In [10]:
y = np.array([[0],
              [1],
              [1],
              [0]])

# Exercise

For the supplied XOR dataset, implement a simple feedforward neural network with one hidden layer with 2 units using sigmoid activation and 1 output unit with sigmoid activation. Train it using the MSE loss function and stochastic gradient descent (SGD), initializing the weights to small random values. Use a learning rate of 0.1, batch size of 2, and train for 10 epochs. After training, report the final weights and the predicted outputs for the XOR inputs.

## Solution

In [18]:
# Sigmoid activation and its derivative
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(z):
    a = sigmoid(z)
    return a * (1 - a)

# Reproducibility
np.random.seed(42)

# Network dimensions
input_size  = 2
hidden_size = 2
output_size = 1

# Weight initialisation (small random values)
W1 = np.random.randn(input_size, hidden_size) * 0.1    # (2, 2)
b1 = np.zeros((hidden_size, 1))                        # (2, 1)
W2 = np.random.randn(hidden_size, output_size) * 0.1   # (2, 1)
b2 = np.zeros((output_size, 1))                        # (1, 1)

W1 = np.random.randn(hidden_size, input_size)
b1 = np.zeros((hidden_size, 1))
W2 = np.random.randn(output_size, hidden_size)
b2 = np.zeros((output_size, 1))

# Hyper-parameters
lr         = 0.1
batch_size = 3
epochs     = 10000
n_samples  = X.shape[0]

# Training loop
for epoch in range(1, epochs + 1):
    # Shuffle data each epoch
    idx = np.random.permutation(n_samples)
    X_s, y_s = X[idx], y[idx]

    epoch_loss = 0.0
    for start in range(0, n_samples, batch_size):
        Xb = X_s[start:start + batch_size]  # (batch, 2)
        yb = y_s[start:start + batch_size]  # (batch, 1)
        m  = Xb.shape[0]

        # --- Forward pass ---
        z1 = (W1 @ Xb.T).T               # (m, 2)
        a1 = sigmoid(z1)           # (m, 2)
        z2 = (W2 @ a1.T).T               # (m, 1)
        a2 = sigmoid(z2)           # (m, 1)

        # MSE loss
        loss = np.mean((a2 - yb) ** 2)
        epoch_loss += loss

        # --- Backward pass ---
        # Output layer
        dL_da2 = 2 * (a2 - yb) / m          # (m, 1)
        dL_dz2 = dL_da2 * sigmoid_deriv(a2).T # (m, 1)
        dL_dW2 = [dL_dz1[:, i].reshape(-1, 1) @ a1[i, :].reshape(1, 2) for i in range(batch_size)]             # (1, 2)

        # Hidden layer
        dL_da1 = (W2 @ dL_dz2).T              # (m, 2)
        dL_dz1 = dL_da1 * sigmoid_deriv(a1) # (m, 2)
        dL_dW1 = [dL_dz1[i, :].reshape(-1, 1) @ Xb[i, :].reshape(1, -1) for i in range(batch_size)]             # (2, 2)

        # --- SGD update ---
        W2 -= lr * np.sum(dL_dW2, axis=0)
        W1 -= lr * np.sum(dL_dW1, axis=0)

    # print(f"Epoch {epoch:2d} | loss: {epoch_loss / (n_samples // batch_size):.6f}")

# --- Final report ---
print("\n=== Final Weights ===")
print(f"W1:\n{W1}")
print(f"W2:\n{W2}")

print("\n=== Predictions vs Ground Truth ===")
z1 = (W1 @ Xb.T).T               # (m, 2)
a1 = sigmoid(z1)           # (m, 2)
z2 = (W2 @ a1).T               # (m, 1)
a2 = sigmoid(z2)           # (m, 1)
preds = a2
for xi, yi, pi in zip(X, y, preds):
    print(f"  Input: {xi}  Target: {yi[0]}  Predicted: {pi[0]:.4f}")

IndexError: index 2 is out of bounds for axis 1 with size 2

Propagation errors
$$
\boldsymbol\delta^{(2)} = \frac{\partial L}{\partial \mathbf{z}^{(2)}} = \frac{\partial L}{\partial \mathbf{h}^{(2)}} \text{diag} \left( \sigma'(\mathbf{z}^{(2)}) \right) = \left( \mathbf{h}^{(2)} - \mathbf{y} \right)^\top \odot \text{diag} \left( \sigma'(\mathbf{z}^{(2)}) \right)^\top = (y - h_2)\sigma'(z_2)
$$
$$
\boldsymbol\delta^{(1)} = \frac{\partial L}{\partial \mathbf{z}^{(1)}} = \left( \mathbf{W}^{(2)} \boldsymbol\delta^{(2)} \right)^\top \odot \text{diag} \left( \sigma'(\mathbf{z}^{(1)}) \right)^\top
$$

In [12]:
dL_dW1

[array([[0., 0.],
        [0., 0.]]),
 array([[0.0338096, 0.       ],
        [0.0338096, 0.       ]])]

In [13]:
dL_dW2

array([[-3.38413361e-02,  1.58771221e-79],
       [-3.38413361e-02,  1.58771221e-79]])

In [14]:
zip(X, y, preds)